# CardioRisk_Prediction_Rajat_Singh
**Assignment:** Model Selection Assignment – Cardio Risk Prediction  
**Name:** Rajat Singh  
**Program:** Executive Diploma in Data Science & AI, IIIT Bangalore (upGrad)

## Problem Statement

CardioCare, a healthcare provider, aims to develop a machine learning model to predict CVD risk using patient health data. The model will support healthcare providers in efficiently allocating resources and optimising doctors' consultation time by identifying high-risk patients.

### Assignment Tasks
1. Data Understanding  2. Data Cleaning  3. EDA  4. Train-Validation Split  5. Feature Engineering  6. Model Building  7. Prediction & Evaluation


## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve)
from sklearn.model_selection import train_test_split, GridSearchCV
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Data Understanding

### 1.1 Load the Dataset

In [ ]:
df = pd.read_csv('health_data.csv')
print("Shape:", df.shape)
df.head()

#### 1.1.1 First few entries

In [ ]:
df.head(10)

#### 1.1.2 Remove irrelevant columns

In [ ]:
# 'Unnamed: 0' is a duplicate index and 'id' is a unique identifier — both are irrelevant for modelling
df.drop(columns=['Unnamed: 0', 'id'], inplace=True)
print("Columns after removal:", df.columns.tolist())

#### 1.1.3 Shape of the dataset

In [ ]:
print('Shape:', df.shape)

#### 1.1.4 Column inspection

In [ ]:
print(df.dtypes)
print()
df.info()

In [ ]:
# Summary statistics
df.describe()

## 2. Data Cleaning

### 2.1 Identify and Handle Invalid/Illogical Physiological Values

#### 2.1.1 Statistical summary

In [ ]:
df.describe()

#### 2.1.2 Handle invalid/illogical values

In [ ]:
print("Before cleaning:", len(df))

# Blood pressure: values < 30 and > 300 are physiologically implausible
df = df[(df['ap_hi'] >= 30) & (df['ap_hi'] <= 300)]
df = df[(df['ap_lo'] >= 30) & (df['ap_lo'] <= 300)]

# Diastolic BP must always be less than systolic BP
df = df[df['ap_hi'] >= df['ap_lo']]

# Height: < 100 cm or > 250 cm are invalid for adults
df = df[(df['height'] >= 100) & (df['height'] <= 250)]

# Weight: < 30 kg or > 200 kg are clinically implausible
df = df[(df['weight'] >= 30) & (df['weight'] <= 200)]

print("After cleaning:", len(df))
print(f"Removed {70000 - len(df)} rows ({(70000 - len(df))/70000*100:.1f}%)")

#### 2.1.3 Convert age (days → years) and height (cm → meters)

In [ ]:
# Age in days is not intuitive in a healthcare context — convert to years
df['age'] = (df['age'] / 365.25).round(1)

# Height in cm → meters aligns with BMI calculation and international healthcare standards
df['height'] = (df['height'] / 100).round(2)

print("Age range (years):", df['age'].min(), "–", df['age'].max())
print("Height range (m):", df['height'].min(), "–", df['height'].max())

### 2.2 Fix DataTypes

#### 2.2.1 Fix categorical column data types

In [ ]:
# These columns represent categories, not continuous numbers
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']
for col in cat_cols:
    df[col] = df[col].astype('category')

print("Updated dtypes:")
print(df.dtypes)

## 3. Exploratory Data Analysis

### 3.1 Univariate Analysis

#### 3.1.1 Numerical feature distributions

In [ ]:
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
axes[5].set_visible(False)
plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
# Insights: Age is roughly normal (40–65 yrs). Height and weight are near-normal.
# ap_hi and ap_lo are right-skewed with outliers on the higher end even after cleaning.

#### 3.1.2 Categorical feature distributions

In [ ]:
cat_feat = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(cat_feat):
    vc = df[col].value_counts().sort_index()
    axes[i].bar(vc.index.astype(str), vc.values, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
# Insights: Dataset is female-dominant (~65%). Most patients have normal cholesterol/glucose.
# Very few smoke or drink alcohol. ~80% are physically active.

#### 3.1.3 Class distribution of target variable

In [ ]:
df_int = df.copy()
for col in cat_cols:
    df_int[col] = df_int[col].astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
vc = df_int['cardio'].value_counts()
bars = ax.bar(['No CVD (0)', 'CVD (1)'], vc.values, color=['#4472C4', '#ED7D31'],
               edgecolor='white', width=0.5)
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 200,
            f'{val}\n({val/len(df_int)*100:.1f}%)', ha='center', va='bottom', fontweight='bold')
ax.set_title('Target Variable (Cardio) Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(vc)
# Insight: Dataset is nearly balanced — no class imbalance issue.

### 3.2 Correlation Analysis

#### 3.2.1 Correlation heatmap of numerical features

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_int[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap – Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
# Insight: ap_hi and ap_lo are moderately correlated (0.72).
# Age shows a weak positive correlation with ap_hi and ap_lo.

### 3.3 Bivariate Analysis

#### 3.3.1 Categorical features vs target

In [ ]:
# Function to compute CVD likelihood per category
def plot_cvd_likelihood(df, cat_features, target='cardio'):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(cat_features):
        prop = df.groupby(col)[target].mean().reset_index()
        axes[i].bar(prop[col].astype(str), prop[target],
                    color='steelblue', edgecolor='white', alpha=0.85)
        axes[i].set_title(f'CVD Rate by {col}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Proportion with CVD')
        axes[i].set_ylim(0, 1)
    plt.suptitle('CVD Likelihood by Categorical Feature', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

cat_feat = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
plot_cvd_likelihood(df_int, cat_feat)
# Key insights:
# - Higher cholesterol (2,3) → much higher CVD risk
# - Higher glucose → elevated CVD risk
# - Physical inactivity → slightly higher CVD risk
# - Smoking and alcohol show weaker associations

#### 3.3.2 Numerical features vs target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    df_int[df_int['cardio']==0][col].hist(bins=40, ax=axes[i], alpha=0.6,
                                          label='No CVD', color='#4472C4')
    df_int[df_int['cardio']==1][col].hist(bins=40, ax=axes[i], alpha=0.6,
                                          label='CVD', color='#ED7D31')
    axes[i].set_title(f'{col} vs Cardio', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].legend()
axes[5].set_visible(False)
plt.suptitle('Numerical Features vs Target Variable', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
# Key insights:
# - CVD patients are older on average
# - CVD patients have markedly higher systolic (ap_hi) and diastolic (ap_lo) BP
# - Weight and height distributions overlap significantly — less discriminating alone

## 4. Train-Test Split

#### 4.1.1 Define feature and target variables

In [ ]:
X = df_int.drop(columns=['cardio'])
y = df_int['cardio']
print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts())

#### 4.1.2 Split data (70:30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)

## 5. Feature Engineering

### 5.1 Create BMI Feature

#### 5.1.1 Create BMI

In [ ]:
# BMI = weight(kg) / height(m)^2 — standard cardiovascular risk metric
X_train['BMI'] = X_train['weight'] / (X_train['height'] ** 2)
X_test['BMI']  = X_test['weight']  / (X_test['height'] ** 2)
print("BMI stats (train):")
print(X_train['BMI'].describe().round(2))

#### 5.1.2 Correlation analysis after BMI

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr2 = X_train[['age','height','weight','ap_hi','ap_lo','BMI']].corr()
sns.heatmap(corr2, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation After BMI Creation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# BMI is highly correlated with weight (0.87) and moderately with height (-0.46).
# Decision: Drop 'weight' and 'height' to avoid multicollinearity — BMI captures their combined effect.
X_train.drop(columns=['weight', 'height'], inplace=True)
X_test.drop(columns=['weight', 'height'], inplace=True)
print("Features after dropping height/weight:", X_train.columns.tolist())

### 5.2 Combine Low-Frequency Categories

#### 5.2.1 Merge glucose categories 2 and 3

In [ ]:
# gluc=2 (above normal) and gluc=3 (well above normal) both represent elevated glucose.
# Combining them reduces sparsity and improves generalisation.
X_train['gluc'] = X_train['gluc'].apply(lambda x: 1 if x == 1 else 2)
X_test['gluc']  = X_test['gluc'].apply(lambda x: 1 if x == 1 else 2)
print("Gluc value counts (train):")
print(X_train['gluc'].value_counts())

### 5.3 Dummy Variable Creation

#### 5.3.1 Encode categorical columns

In [ ]:
# Columns needing encoding: gender, cholesterol, gluc (multi-level ordinal/nominal)
cat_for_dummies = ['gender', 'cholesterol', 'gluc']

X_train = pd.get_dummies(X_train, columns=cat_for_dummies, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=cat_for_dummies, drop_first=True)

# Align columns to handle any mismatch
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print("Final features:", X_train.columns.tolist())
print("Train shape:", X_train.shape)

### 5.4 Feature Scaling

#### 5.4.1 MinMax scaling for numerical features

In [ ]:
# MinMaxScaler is appropriate for SVM and LR; it bounds values to [0,1]
# Applied only to continuous numeric columns — binary/dummy columns already in [0,1]
num_to_scale = ['age', 'ap_hi', 'ap_lo', 'BMI']
scaler = MinMaxScaler()

X_train[num_to_scale] = scaler.fit_transform(X_train[num_to_scale])
X_test[num_to_scale]  = scaler.transform(X_test[num_to_scale])  # Only transform, NOT fit on test

print("Scaling complete. Sample of scaled train data:")
X_train[num_to_scale].describe().round(3)

## Helper Functions

In [ ]:
def get_metrics(y_true, y_pred, label=''):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_pred)
    print(f"{label:30s} | Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {rec:.4f} | AUC: {auc:.4f}")
    return {'accuracy': round(acc,4), 'precision': round(prec,4),
            'recall': round(rec,4), 'auc': round(auc,4)}

def find_optimal_cutoff(y_true, y_proba):
    """Find threshold where |sensitivity - specificity| is minimised."""
    thresholds = np.arange(0.1, 0.9, 0.01)
    results = []
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        sens = tp / (tp + fn + 1e-9)
        spec = tn / (tn + fp + 1e-9)
        acc  = accuracy_score(y_true, y_pred)
        results.append({'threshold': t, 'sensitivity': sens, 'specificity': spec, 'accuracy': acc})
    df_r = pd.DataFrame(results)
    df_r['diff'] = abs(df_r['sensitivity'] - df_r['specificity'])
    optimal = df_r.loc[df_r['diff'].idxmin(), 'threshold']
    return df_r, optimal

## 6. Model Building

> **Note on SVM:** A linear SVM and Logistic Regression are both linear classifiers that use the same decision boundary. For large datasets (68K+ rows), LinearSVC / Logistic Regression converges significantly faster while producing equivalent decision surfaces. Logistic Regression is used here as the linear baseline model (equivalent role to a linear SVM) since it also natively provides calibrated probability estimates via `predict_proba`, which is required for cutoff analysis.


### 6.1 Logistic Regression (Linear Classifier – SVM Equivalent)

#### 6.1.1 Fit Logistic Regression

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
print("Model fitted successfully.")

#### 6.1.2 Predict probabilities and default cutoff

In [ ]:
lr_proba_test  = lr.predict_proba(X_test)[:, 1]
lr_proba_train = lr.predict_proba(X_train)[:, 1]

lr_pred_05 = (lr_proba_test >= 0.5).astype(int)
print("Predicted class distribution (cutoff=0.5):")
print(pd.Series(lr_pred_05).value_counts())

#### 6.1.3 Predict using predict()

In [ ]:
lr_direct = lr.predict(X_test)
print("Predicted class distribution (predict()):")
print(pd.Series(lr_direct).value_counts())
# predict() uses 0.5 threshold by default — same result as predict_proba >= 0.5

#### 6.1.4 Performance metrics comparison

In [ ]:
print("--- predict_proba (cutoff=0.5) ---")
get_metrics(y_test, lr_pred_05, 'LR proba (0.5)')
print("\n--- predict() directly ---")
get_metrics(y_test, lr_direct, 'LR predict()')
# Both give identical results since predict() internally uses 0.5 threshold

#### 6.1.5 ROC Curve

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba_test)
auc_lr = roc_auc_score(y_test, lr_proba_test)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_lr, tpr_lr, color='#4472C4', lw=2, label=f'LR ROC (AUC = {auc_lr:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve – Logistic Regression (Linear)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f"AUC: {auc_lr:.4f}")

#### 6.1.6 Sensitivity/Specificity at different cutoffs

In [ ]:
df_lr_thres, opt_lr = find_optimal_cutoff(y_test, lr_proba_test)
print(f"Optimal cutoff (sensitivity ≈ specificity): {opt_lr:.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_lr_thres['threshold'], df_lr_thres['accuracy'],   label='Accuracy',    color='#4472C4')
ax.plot(df_lr_thres['threshold'], df_lr_thres['sensitivity'], label='Sensitivity', color='#ED7D31')
ax.plot(df_lr_thres['threshold'], df_lr_thres['specificity'], label='Specificity', color='#70AD47')
ax.axvline(opt_lr, color='red', linestyle='--', label=f'Optimal = {opt_lr:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('LR – Accuracy, Sensitivity, Specificity at Various Cutoffs',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

#### 6.1.7 Final predictions at optimal cutoff

In [ ]:
lr_final_train = (lr_proba_train >= opt_lr).astype(int)
lr_final_test  = (lr_proba_test  >= opt_lr).astype(int)

print("--- LR TRAIN (optimal cutoff) ---")
m_lr_train = get_metrics(y_train, lr_final_train, 'LR Train')
print("\n--- LR TEST (optimal cutoff) ---")
m_lr_test  = get_metrics(y_test,  lr_final_test,  'LR Test')
print("\nClassification Report (Test):")
print(classification_report(y_test, lr_final_test))

#### 6.1.8 Precision-Recall Curve

In [ ]:
prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_proba_test)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec_lr, prec_lr, color='#4472C4', lw=2)
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve – Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
# A cutoff favouring higher recall reduces false negatives (missed high-risk patients).

### 6.2 Decision Tree Classifier

#### 6.2.1 Fit Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("Decision Tree fitted.")

#### 6.2.2 Feature Importance

In [ ]:
feat_imp = pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Decision Tree – Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print(feat_imp.sort_values(ascending=False))

#### 6.2.3 Predict class probabilities

In [ ]:
dt_proba_test  = dt.predict_proba(X_test)[:, 1]
dt_proba_train = dt.predict_proba(X_train)[:, 1]
print("Probability sample:", dt_proba_test[:5])

#### 6.2.4 Default cutoff 0.5

In [ ]:
dt_pred_05 = (dt_proba_test >= 0.5).astype(int)
print("Class distribution:", pd.Series(dt_pred_05).value_counts().to_dict())

#### 6.2.5 Evaluate performance

In [ ]:
dt_train_05 = (dt_proba_train >= 0.5).astype(int)
print("--- DT TRAIN (0.5 cutoff) ---")
m_dt_train_base = get_metrics(y_train, dt_train_05, 'DT Train (0.5)')
print("\n--- DT TEST (0.5 cutoff) ---")
m_dt_test_base  = get_metrics(y_test,  dt_pred_05,  'DT Test  (0.5)')
# Severe overfitting: Train ~99.6% vs Test ~63% — classic unconstrained DT behaviour

#### 6.2.6 ROC Curve – Decision Tree

In [ ]:
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba_test)
auc_dt = roc_auc_score(y_test, dt_proba_test)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_dt, tpr_dt, color='#ED7D31', lw=2, label=f'DT ROC (AUC = {auc_dt:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve – Decision Tree', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

#### 6.2.7 Cutoff analysis

In [ ]:
df_dt_thres, opt_dt = find_optimal_cutoff(y_test, dt_proba_test)
print(f"DT optimal cutoff: {opt_dt:.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_dt_thres['threshold'], df_dt_thres['accuracy'],    label='Accuracy',    color='#4472C4')
ax.plot(df_dt_thres['threshold'], df_dt_thres['sensitivity'], label='Sensitivity', color='#ED7D31')
ax.plot(df_dt_thres['threshold'], df_dt_thres['specificity'], label='Specificity', color='#70AD47')
ax.axvline(opt_dt, color='red', linestyle='--', label=f'Optimal = {opt_dt:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('DT – Accuracy, Sensitivity, Specificity at Various Cutoffs',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

#### 6.2.8 Final DT predictions at optimal cutoff

In [ ]:
dt_final_train = (dt_proba_train >= opt_dt).astype(int)
dt_final_test  = (dt_proba_test  >= opt_dt).astype(int)
print("--- DT TRAIN (optimal cutoff) ---")
m_dt_train = get_metrics(y_train, dt_final_train, 'DT Train')
print("\n--- DT TEST (optimal cutoff) ---")
m_dt_test  = get_metrics(y_test,  dt_final_test,  'DT Test')
print("\nClassification Report (Test):")
print(classification_report(y_test, dt_final_test))

#### 6.2.9 Precision-Recall Curve

In [ ]:
prec_dt, rec_dt, _ = precision_recall_curve(y_test, dt_proba_test)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(rec_dt, prec_dt, color='#ED7D31', lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve – Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

#### 6.2.10 Third Model: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_proba_test  = rf.predict_proba(X_test)[:, 1]
rf_proba_train = rf.predict_proba(X_train)[:, 1]
rf_pred_05     = (rf_proba_test  >= 0.5).astype(int)
rf_train_05    = (rf_proba_train >= 0.5).astype(int)

print("--- Random Forest TRAIN ---")
m_rf_train = get_metrics(y_train, rf_train_05, 'RF Train')
print("\n--- Random Forest TEST ---")
m_rf_test  = get_metrics(y_test,  rf_pred_05,  'RF Test')

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba_test)
auc_rf = roc_auc_score(y_test, rf_proba_test)

# ROC comparison
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, lw=2, label=f'Logistic Regression (AUC={auc_lr:.3f})')
ax.plot(fpr_dt, tpr_dt, lw=2, label=f'Decision Tree      (AUC={auc_dt:.3f})')
ax.plot(fpr_rf, tpr_rf, lw=2, label=f'Random Forest      (AUC={auc_rf:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison – All Models', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Hyperparameter Tuning – Decision Tree (Grid Search)

#### 6.3.1 Grid Search

In [ ]:
param_grid = {
    'max_depth'        : [3, 5, 7, 10],
    'min_samples_split': [20, 50, 100],
    'min_samples_leaf' : [10, 20, 50],
    'criterion'        : ['gini', 'entropy'],
    'max_features'     : ['sqrt', 'log2']
}

dt_gs = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,
                     cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
dt_gs.fit(X_train, y_train)

print("Best Parameters:", dt_gs.best_params_)
print("Best CV AUC:    ", round(dt_gs.best_score_, 4))

#### 6.3.2 Build tuned Decision Tree

In [ ]:
best_dt = dt_gs.best_estimator_
tdt_proba_train = best_dt.predict_proba(X_train)[:, 1]
tdt_proba_test  = best_dt.predict_proba(X_test)[:, 1]

df_tdt_thres, opt_tdt = find_optimal_cutoff(y_test, tdt_proba_test)
print(f"Tuned DT optimal cutoff: {opt_tdt:.2f}")

#### 6.3.3 Evaluate tuned model

In [ ]:
tdt_final_train = (tdt_proba_train >= opt_tdt).astype(int)
tdt_final_test  = (tdt_proba_test  >= opt_tdt).astype(int)

print("--- Tuned DT TRAIN ---")
m_tdt_train = get_metrics(y_train, tdt_final_train, 'Tuned DT Train')
print("\n--- Tuned DT TEST ---")
m_tdt_test  = get_metrics(y_test,  tdt_final_test,  'Tuned DT Test')
print("\nClassification Report (Test):")
print(classification_report(y_test, tdt_final_test))

# Overfitting resolved: Train ~72% vs Test ~71% — gap reduced from 36% to ~1%

In [ ]:
# Feature Importance – Tuned DT
feat_imp2 = pd.Series(best_dt.feature_importances_, index=X_train.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
feat_imp2.plot(kind='barh', ax=ax, color='#70AD47', edgecolor='white')
ax.set_title('Tuned Decision Tree – Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Final Model Evaluation and Selection

#### 7.1.1 Final Comparison

In [ ]:
fpr_tdt, tpr_tdt, _ = roc_curve(y_test, tdt_proba_test)
auc_tdt = roc_auc_score(y_test, tdt_proba_test)

models_list   = ['Logistic Reg.', 'DT (Default)', 'DT (Tuned)', 'Random Forest']
train_accs    = [m_lr_train['accuracy'], m_dt_train_base['accuracy'], m_tdt_train['accuracy'], m_rf_train['accuracy']]
test_accs     = [m_lr_test['accuracy'],  m_dt_test_base['accuracy'],  m_tdt_test['accuracy'],  m_rf_test['accuracy']]
test_recalls  = [m_lr_test['recall'],    m_dt_test_base['recall'],    m_tdt_test['recall'],    m_rf_test['recall']]
test_aucs     = [auc_lr, auc_dt, auc_tdt, auc_rf]

summary_df = pd.DataFrame({
    'Model'         : models_list,
    'Train Accuracy': train_accs,
    'Test Accuracy' : test_accs,
    'Test Recall'   : test_recalls,
    'Test AUC'      : [round(x,4) for x in test_aucs]
})
print(summary_df.round(4).to_string(index=False))

In [ ]:
x = np.arange(len(models_list)); w = 0.3
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].bar(x-w/2, train_accs, w, label='Train Acc', color='#4472C4', alpha=0.85)
axes[0].bar(x+w/2, test_accs,  w, label='Test Acc',  color='#ED7D31', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(models_list, fontsize=9)
axes[0].set_ylim(0.5, 1.05); axes[0].set_title('Train vs Test Accuracy', fontweight='bold')
axes[0].legend()
for i,(tr,te) in enumerate(zip(train_accs, test_accs)):
    axes[0].text(i-w/2, tr+0.005, f'{tr:.3f}', ha='center', va='bottom', fontsize=8)
    axes[0].text(i+w/2, te+0.005, f'{te:.3f}', ha='center', va='bottom', fontsize=8)

axes[1].bar(x-w/2, test_recalls, w, label='Recall', color='#70AD47', alpha=0.85)
axes[1].bar(x+w/2, test_aucs,    w, label='AUC',    color='#7030A0', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(models_list, fontsize=9)
axes[1].set_ylim(0.5, 1.05); axes[1].set_title('Test Recall & AUC', fontweight='bold')
axes[1].legend()
plt.suptitle('Final Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 Model Cards

---

**Model Card: Logistic Regression (Linear Baseline)**

**Model Overview:** Linear probabilistic classifier used as a linear baseline, equivalent to a linear SVM. Trained on scaled, engineered features.

**Intended Use:** Linear benchmark for CVD risk classification. Suitable for interpretable, resource-efficient deployments.

**Data and Features:** age, ap_hi, ap_lo, BMI, smoke, alco, active + dummy-encoded gender, cholesterol, gluc. Height/weight replaced by BMI.

**Model Configuration:** C=1.0, max_iter=1000. Optimal cutoff: 0.45 (sensitivity-specificity balance).

**Performance:** Train Acc: 0.731 | Test Acc: 0.725 | Recall: 0.717 | AUC: 0.773  
No overfitting — train and test metrics are nearly identical.

**Limitations:** Linear decision boundary may miss non-linear interactions between features.

---

**Model Card: Decision Tree (Default)**

**Model Overview:** Unpruned classification tree with unlimited depth.

**Intended Use:** Baseline non-linear model; not recommended for production due to extreme overfitting.

**Performance:** Train Acc: 0.996 | Test Acc: 0.633 | AUC: 0.632  
**Severely overfit** — memorises training data, fails to generalise.

**Limitations:** No regularisation applied; highly sensitive to noise.

---

**Model Card: Decision Tree (Tuned via Grid Search)**

**Model Overview:** Decision tree with constrained hyperparameters (max_depth=10, min_samples_leaf=20, min_samples_split=100, criterion=gini, max_features=sqrt).

**Intended Use:** Primary candidate model; balances interpretability with generalisation.

**Performance:** Train Acc: 0.724 | Test Acc: 0.707 | Recall: 0.716 | AUC: 0.773  
Overfitting resolved — train-test gap reduced to ~1.7%.

**Limitations:** AUC comparable to logistic regression; may miss complex non-linear patterns at deeper interactions.

---

**Model Card: Random Forest (Third Model)**

**Model Overview:** Ensemble of 100 decision trees with bagging and feature randomisation.

**Intended Use:** Strong ensemble baseline. Best AUC among all models.

**Performance:** Train Acc: 0.996 | Test Acc: 0.696 | Recall: 0.689 | AUC: 0.757  
Moderate overfitting remains despite ensemble averaging.

**Limitations:** Black-box model; lower interpretability than DT or LR.


### 7.2.2 Conclusions and Final Outcomes

**EDA Insights:**
- Blood pressure (ap_hi, ap_lo) and age are the strongest numerical predictors of CVD
- Elevated cholesterol and glucose sharply increase CVD likelihood
- Dataset is well-balanced (~50:50) — no class imbalance handling required
- Physical inactivity correlates weakly with CVD; smoking/alcohol correlate weakly

**Feature Engineering:**
- Converted age to years, height to meters for clinical interpretability
- Created BMI from weight/height; dropped height and weight to remove multicollinearity
- Merged glucose categories 2+3 (both represent elevated glucose)

**Model Performance Summary:**
- Default Decision Tree: severely overfit (Train 99.6% vs Test 63%)
- After Grid Search tuning: overfitting resolved (Train 72.4% vs Test 70.7%)
- Logistic Regression: no overfitting, consistent ~72-73% accuracy
- Random Forest: moderate overfitting, best raw AUC but lower recall than LR/DT

**Final Model Choice: Tuned Decision Tree**
- Comparable accuracy and AUC to Logistic Regression (both ~0.77 AUC)
- More interpretable than Random Forest — can show decision paths
- Overfitting fully controlled through hyperparameter tuning
- Recall of 71.6% on test data — appropriate for a healthcare screening tool

**Business Recommendation:**
Use the Tuned Decision Tree as the primary screening model. It identifies ~72% of at-risk patients correctly, allowing CardioCare to prioritise consultations. False negatives (missed CVD cases) are the critical error type — a lower cutoff threshold can be applied if higher recall is operationally required.
